# Qwen3-Embedding-4B Fine-Tune trên Kaggle (2×T4) — train + warmup validation + **HNSW Indexing**

Fine-tune [`Qwen/Qwen3-Embedding-4B`](https://huggingface.co/Qwen/Qwen3-Embedding-4B) bằng **LoRA**
(`peft`, rank=16 mặc định) + **contrastive learning với in-batch negatives** — y hệt phương pháp của
`vietlegal-tune-kaggle-hnsw.ipynb` (chỉ đổi `MODEL_NAME` + hạ batch size cho model lớn hơn), không
cần BM25/hard-negative mining. Set `USE_LORA = False` ở cell config để quay lại full fine-tune,
nhưng với 4B tham số thì **không khuyến nghị** — full fine-tune không có gradient checkpointing đủ
mạnh sẽ tràn VRAM T4 (15GB) gần như chắc chắn.

**Quy trình đơn giản**: fine-tune model → encode corpus → **BUILD HNSW INDEX** → truy vấn **dùng HNSW** → lấy top-5 `ctx_id`/câu →
tính **Recall@5** + **Precision** (đúng công thức `scoring.py`) trên `warmup.json`.

**Tối ưu hóa tốc độ**: Thay vì tính cosine similarity với tất cả corpus (brute-force O(n)), HNSW xây dựng index hierarchical graph → truy vấn O(log n). Trên 400k chunks, truy vấn **nhanh 10-50x**.

**Khác biệt so với `vietlegal-tune-kaggle-hnsw.ipynb` (fine-tune harrier 0.6B)**, vì Qwen3-Embedding-4B
nặng hơn ~6.7× số tham số (36 layer, hidden 2560 so với hidden ~1024 của harrier):

| | harrier 0.6B | Qwen3-Embedding-4B |
|---|---|---|
| `BATCH_SIZE` (encode corpus/query) | 128 | **32** |
| `Q_BATCH` (brute-force fallback) | 256 | **128** |
| `TRAIN_BATCH_SIZE` | 32 | **4** |
| `NUM_EPOCHS` | 5 | **3** (mỗi epoch tốn nhiều compute hơn hẳn — hạ số epoch để vẫn chạy nổi trong 1 phiên Kaggle) |
| Embedding dim | 1024 | **2560** (tự động dò ở `corpus_emb.shape[1]`, không cần sửa gì thêm) |

Nếu vẫn OOM ở batch trên, hạ tiếp `TRAIN_BATCH_SIZE=2` hoặc `LORA_R=8` trước khi tắt gradient
checkpointing hay tắt LoRA.

- **Data**: `/kaggle/input/datasets/nhtclone/legal-data` (cấu trúc hệt local, dùng chung với notebook harrier).
- **Cache baseline**: `/kaggle/input/datasets/nhtclone/qwen3-state` (dataset **riêng**, không dùng chung
  `vietlegal-state` của harrier — pooling giống nhau về hình thức (last-token) nhưng `MODEL_NAME` khác
  nên `_legal_state_meta_ok()` sẽ luôn coi cache của harrier không khớp và bỏ qua).
  Nếu có sẵn corpus embedding/kết quả truy vấn của Qwen3-Embedding-4B **trước khi fine-tune**, notebook
  sẽ **load thẳng thay vì encode lại** để tiết kiệm thời gian.
- **2× GPU T4 (15GB)**: `torch.nn.DataParallel(model)`, model + embedding ở **fp16**.
- **Huấn luyện**: **3 epoch** (xem bảng trên), in-batch negative InfoNCE (giống `MultipleNegativesRankingLoss`),
  `AdamW` + `CosineAnnealingLR`.
- **Lưu checkpoint**: mỗi khi **Recall@5 trên warmup ≥ 0.8292 + 0.005 = 0.8342**.
  ⚠️ `BASE_RECALL_REF=0.8292` là **mốc tham chiếu chéo model** (Recall@5 của `vnlegal-lal`, mượn lại y
  hệt từ notebook harrier) — **không phải** baseline zero-shot thật của Qwen3-Embedding-4B, vì model
  này chưa từng được đo trên bộ dữ liệu này. Khuyến nghị: lần chạy **đầu tiên** nên đặt
  `SKIP_BASELINE = False` để đo baseline thật (~2-3h, chỉ cần đo 1 lần rồi cache), sau đó mới bật lại
  `SKIP_BASELINE = True` với `BASE_RECALL_REF` chỉnh theo số vừa đo được.

**Output** (`/kaggle/working/`):
- `qwen3_finetuned/epoch{N}_recall{R}.pth` (hoặc thư mục adapter nếu LoRA), `qwen3_finetuned/best_adapter` — checkpoint đạt ngưỡng
- `qwen3_finetuned/config.json` — cấu hình + kết quả cuối
- `warmup_finetuned_best.json`, `train_finetuned_best.json` — kết quả truy vấn của checkpoint tốt nhất
- `finetuning_log.json` — loss + Recall@5/Precision mỗi epoch (kể cả baseline epoch 0)
- **`*.hnsw` index files** — HNSW index (lưu để tái sử dụng nhanh hơn)

## 1. Cài đặt thư viện

> Qwen3 (kiến trúc dùng cho cả LLM và embedding) chỉ được hỗ trợ từ `transformers>=4.51.0` —
> pin cận dưới, không chỉ cận trên như bản harrier. Kaggle cần bật Internet trong Settings.
> **Thêm hnswlib để indexing.**

In [ ]:
!pip install -q "transformers>=4.51.0,<5" torch tqdm hnswlib "peft>=0.10.0"

## 2. Cấu hình

In [ ]:
import os

# Phải set TRƯỚC khi import torch (cell sau) để có tác dụng. Giảm OOM do phân mảnh bộ nhớ CUDA.
# PyTorch đổi tên biến này qua vài phiên bản nên set cả hai cho chắc.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

LEGAL_DATA  = "/kaggle/input/datasets/nhtclone/legal-data"    # input (read-only)
LEGAL_STATE = "/kaggle/input/datasets/nhtclone/qwen3-state"   # dataset RIÊNG, không dùng chung vietlegal-state
OUT_DIR     = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
MODEL_OUT_DIR = os.path.join(OUT_DIR, "qwen3_finetuned")
os.makedirs(MODEL_OUT_DIR, exist_ok=True)

CHUNK_DIR = os.path.join(LEGAL_DATA, "data", "chunk-context")
QUERY_FILES = {
    "warmup":      os.path.join(LEGAL_DATA, "data", "warmup.json"),
    "train": os.path.join(LEGAL_DATA, "data", "train.json"),
}

MODEL_NAME     = "Qwen/Qwen3-Embedding-4B"
QUERY_INSTRUCT = ("Instruct: Given a Vietnamese legal question, retrieve relevant legal passages "
                  "that answer the question\nQuery: ")
MAX_SEQ_LEN    = 512
USE_FP16       = True
BATCH_SIZE     = 32    # hạ từ 128 (harrier 0.6B) — Qwen3-Embedding-4B nặng hơn ~6.7x
QUERY_BATCH    = 32
Q_BATCH        = 128   # chỉ dùng trong fallback brute-force
TOP_SAVE       = 200

# --- LoRA (Low-Rank Adaptation) ---
USE_LORA            = True      # False = full fine-tune (giữ hành vi cũ)
LORA_R              = 16        # rank
LORA_ALPHA          = 32        # scaling = alpha / r
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = "all-linear"   # áp LoRA lên mọi Linear layer — không cần biết tên module cụ thể của Qwen3

# --- Fine-tune ---
NUM_EPOCHS       = 3    # hạ từ 5 (harrier) — mỗi epoch tốn nhiều compute hơn hẳn với 4B tham số
LEARNING_RATE    = 2e-4 if USE_LORA else 2e-5   # LoRA cần LR cao hơn full fine-tune
                                                  # (update qua ma trận rank-thấp có biên độ nhỏ hơn nhiều)
TRAIN_BATCH_SIZE = 4    # 4B tham số + fp16 ~8GB VRAM chỉ riêng trọng số/GPU (DataParallel replica đầy đủ,
                         # không shard) — batch 4 + gradient checkpointing là mốc an toàn trên T4 15GB.
                         # OOM thì hạ tiếp xuống 2 hoặc LORA_R=8 trước khi tắt gradient checkpointing.
TEMPERATURE      = 0.05

# --- Ngưỡng lưu checkpoint ---
BASE_RECALL_REF  = 0.8292
MIN_IMPROVEMENT  = 0.005
SAVE_THRESHOLD   = BASE_RECALL_REF + MIN_IMPROVEMENT

# ⚠️ BASE_RECALL_REF là mốc tham chiếu CHÉO MODEL (từ vnlegal-lal), KHÔNG PHẢI baseline
# zero-shot thật của Qwen3-Embedding-4B — model này chưa từng được đo trên bộ dữ liệu này.
# Lần chạy ĐẦU TIÊN nên đặt False để đo baseline thật (~2-3h), rồi cache lại vào qwen3-state
# để các lần sau có thể bật lại True mà không phải đo lại.
SKIP_BASELINE    = False

# --- **HNSW Parameters** ---
USE_HNSW         = True           # Bật/tắt HNSW indexing
HNSW_M           = 16             # Max connections per node (16-48 typical)
HNSW_EF_CONST    = 200            # ef_construction (400-1000 cho chất lượng cao)
HNSW_EF_SEARCH   = 200            # ef search (>=k, để tìm top-k chính xác)
HNSW_EF_SEARCH   = max(HNSW_EF_SEARCH, TOP_SAVE + 100)  # ef phải > k=TOP_SAVE để có đủ dư địa tìm kiếm,
                                                          # bằng đúng k (200==200) làm giảm độ chính xác top-k

print("Legal data :", LEGAL_DATA)
print("Legal state:", LEGAL_STATE)
print("Out dir    :", OUT_DIR)
print(f"Model      : {MODEL_NAME} | MAX_SEQ_LEN={MAX_SEQ_LEN} | USE_FP16={USE_FP16}")
print(f"Train      : {NUM_EPOCHS} epoch | batch={TRAIN_BATCH_SIZE} | lr={LEARNING_RATE}")
print(f"LoRA       : {'ENABLED r=' + str(LORA_R) + ' alpha=' + str(LORA_ALPHA) if USE_LORA else 'DISABLED (full fine-tune)'}")
print(f"Save khi Recall@5(warmup) >= {SAVE_THRESHOLD:.4f}")
print(f"HNSW       : {'ENABLED' if USE_HNSW else 'DISABLED'} | M={HNSW_M} | ef_const={HNSW_EF_CONST} | ef_search={HNSW_EF_SEARCH}")

## 3. Load harrier (fp16) + hàm encode (DataParallel 2×T4)

In [ ]:
import json, gc, pickle, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMB_DTYPE = torch.float16 if USE_FP16 else torch.float32
N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("Device:", DEVICE, "| GPUs:", N_GPU, "| EMB_DTYPE:", EMB_DTYPE)

print(f"Đang load {MODEL_NAME} (fp16)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="left")
# Qwen3-Embedding khuyến nghị padding bên trái cho last-token pooling (khớp cách HF công bố dùng
# model này) — last_token_pool() bên dưới tự dò padding side qua attention_mask nên vẫn đúng dù
# lỡ đổi lại "right", nhưng đặt tường minh ở đây cho khớp đúng khuyến nghị gốc.
base_model = AutoModel.from_pretrained(MODEL_NAME, dtype=EMB_DTYPE, trust_remote_code=True,
                                        attn_implementation="sdpa").to(DEVICE)

if USE_LORA:
    from peft import LoraConfig, TaskType, get_peft_model

    # Kaggle cài sẵn torchao==0.10.0, thấp hơn mức peft yêu cầu (0.16.0). peft dò
    # is_torchao_available() cho MỌI Linear layer khi inject LoRA -- kể cả khi ta
    # không dùng torchao/quantization -- và ném ImportError thay vì trả False khi
    # version không đạt. Vô hiệu hoá nhánh dò này vì model chạy fp16 thường.
    import peft.tuners.lora.torchao as _lora_torchao
    _lora_torchao.is_torchao_available = lambda: False

    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES, bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
    )
    base_model = get_peft_model(base_model, lora_config)
    # Base đóng băng ở fp16; nâng riêng adapter LoRA (A/B) lên fp32 để optimizer update ổn định —
    # đúng công thức mixed-precision LoRA chuẩn (base fp16/bf16 đóng băng + adapter fp32 huấn luyện).
    # peft tự cast input về dtype của lora_A trước khi nhân, nên việc trộn dtype này an toàn.
    for name, param in base_model.named_parameters():
        if param.requires_grad:
            param.data = param.data.float()
    base_model.print_trainable_parameters()

    # Gradient checkpointing: cắt lớn activation memory (đổi lấy ~20-30% chậm hơn) — cần thiết
    # vì Qwen3-Embedding-4B có ~36 layer, activation của toàn bộ layer bị giữ lại cho backward nếu
    # không checkpoint sẽ tràn VRAM T4 (14.56GB) gần như chắc chắn dù đã LoRA. enable_input_require_grads()
    # bắt buộc đi kèm
    # vì base bị đóng băng — nếu không, chuỗi backward của checkpointing đứt tại input embedding.
    base_model.gradient_checkpointing_enable()
    base_model.enable_input_require_grads()

model = nn.DataParallel(base_model) if N_GPU > 1 else base_model
print(f"Model sẵn sàng ({'DataParallel ' + str(N_GPU) + ' GPU' if N_GPU > 1 else '1 GPU/CPU'}"
      f"{' + LoRA r=' + str(LORA_R) if USE_LORA else ''}).")


def get_trainable_model():
    """Model PEFT/base thật, bỏ DataParallel wrapper — để gọi save_pretrained/merge_and_unload."""
    return model.module if isinstance(model, nn.DataParallel) else model


def last_token_pool(last_hidden, attention_mask):
    """Pooling theo token cuối."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden[:, -1]
    seq_lengths = attention_mask.sum(dim=1) - 1
    batch_idx = torch.arange(last_hidden.shape[0], device=last_hidden.device)
    return last_hidden[batch_idx, seq_lengths]


@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE, desc=None):
    """Encode danh sách văn bản -> numpy fp16, đã L2-normalize."""
    model.eval()
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc, leave=False):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_SEQ_LEN, return_tensors="pt").to(DEVICE)
        hidden = model(**enc).last_hidden_state
        emb = last_token_pool(hidden, enc["attention_mask"])
        emb = F.normalize(emb.float(), p=2, dim=1)
        out.append(emb.half().cpu().numpy() if USE_FP16 else emb.cpu().numpy())
    return np.concatenate(out, axis=0)


def encode_forward(texts):
    """Encode (TRAINING) danh sách văn bản -> tensor float32 L2-normalize, trên GPU."""
    enc = tokenizer(texts, padding=True, truncation=True,
                    max_length=MAX_SEQ_LEN, return_tensors="pt").to(DEVICE)
    hidden = model(**enc).last_hidden_state
    emb = last_token_pool(hidden, enc["attention_mask"])
    return F.normalize(emb.float(), p=2, dim=1)

## 4. Load corpus chunk

In [ ]:
def corpus_signature():
    files = sorted(f for f in os.listdir(CHUNK_DIR) if f.endswith(".json"))
    total = sum(os.path.getsize(os.path.join(CHUNK_DIR, f)) for f in files)
    return (len(files), total)


files = sorted(f for f in os.listdir(CHUNK_DIR) if f.endswith(".json"))
sig = corpus_signature()

all_chunk_texts = []
ctx_ids_list, chunk_idx_list = [], []
ctx_id_to_indices = {}

for f in tqdm(files, desc="Load chunks"):
    s = json.load(open(os.path.join(CHUNK_DIR, f), encoding="utf-8"))
    ctx_id = int(s["id"])
    for k, text in s["chunk"].items():
        if not text.strip():
            continue
        idx = len(all_chunk_texts)
        all_chunk_texts.append(text)
        ctx_ids_list.append(ctx_id)
        chunk_idx_list.append(int(k))
        ctx_id_to_indices.setdefault(ctx_id, []).append(idx)

ctx_ids = np.asarray(ctx_ids_list, dtype=np.int32)
chunk_idx = np.asarray(chunk_idx_list, dtype=np.int32)
print(f"Tổng chunk: {len(all_chunk_texts)} | Văn bản: {len(ctx_id_to_indices)}")

## 5. **HNSW Indexing - Build & Query**

In [ ]:
import hnswlib


def build_hnsw_index(corpus_emb, dim=2560, desc="Build HNSW"):
    """
    Build HNSW index từ corpus embeddings.
    corpus_emb: (n_corpus, dim) numpy array, chuẩn hóa L2. `dim` mặc định 2560 = hidden size của
    Qwen3-Embedding-4B, nhưng luôn bị ghi đè bằng corpus_emb.shape[1] ở nơi gọi nên không cần sửa
    nếu đổi sang model khác dim khác.
    Returns: hnswlib.Index object, sẵn sàng để query
    """
    print(f"[HNSW] Building index: {corpus_emb.shape[0]} vectors, dim={dim}")
    
    index = hnswlib.Index(space='cosine', dim=dim)
    index.init_index(max_elements=corpus_emb.shape[0], ef_construction=HNSW_EF_CONST, M=HNSW_M)
    index.add_items(corpus_emb, np.arange(corpus_emb.shape[0]))
    
    index.ef = HNSW_EF_SEARCH
    print(f"[HNSW] Index built successfully (M={HNSW_M}, ef_construction={HNSW_EF_CONST}, ef_search={HNSW_EF_SEARCH})")
    return index


def query_hnsw_index(index, query_embs, k):
    """
    Query HNSW index với batch query embeddings.
    query_embs: (n_queries, dim) numpy array, L2-normalized cosine
    k: số kết quả trả về per query
    Returns: (labels, distances) - indices và cosine DISTANCES (1 - cosine similarity,
    càng nhỏ càng giống nhau) — hnswlib space='cosine' trả về distance, không phải similarity.
    """
    labels, distances = index.knn_query(query_embs, k=k)
    return labels, distances


def retrieve_dataset_with_hnsw(query_path, corpus_emb, k_save=TOP_SAVE, desc="Retrieve"):
    """
    Truy vấn dataset dùng HNSW index thay vì brute-force cosine similarity.
    """
    print(f"[HNSW] Building index for retrieval...")
    t_build = time.time()
    index = build_hnsw_index(corpus_emb, dim=corpus_emb.shape[1])
    print(f"[HNSW] Index built in {time.time() - t_build:.1f}s")
    
    queries = json.load(open(query_path, encoding="utf-8"))
    qids = sorted(queries.keys())
    q_texts = [QUERY_INSTRUCT + queries[q]["question"] for q in qids]
    q_embs = encode_texts(q_texts, batch_size=QUERY_BATCH, desc=desc)
    
    out = {}
    k_save = min(k_save, corpus_emb.shape[0])
    
    print(f"[HNSW] Querying {len(qids)} queries, k={k_save}...")
    t_query = time.time()
    labels, distances = query_hnsw_index(index, q_embs, k=k_save)
    print(f"[HNSW] Query completed in {time.time() - t_query:.1f}s")
    
    for j, qid in enumerate(qids):
        out[qid] = {
            "question": queries[qid]["question"],
            "results": [{
                "ctx_id": int(ctx_ids[int(idx)]),
                "chunk": int(chunk_idx[int(idx)]),
                # hnswlib cosine space trả distance = 1 - similarity; đảo lại để "score" nghĩa là
                # cosine similarity (higher = better), khớp semantics với retrieve_dataset_brute_force.
                "score": round(1.0 - float(distances[j, t]), 4)
            } for t, idx in enumerate(labels[j])],
        }
    
    del index
    gc.collect()
    return out

## 5b. Hàm dùng chung: encode corpus / tính metrics (giữ nguyên)

In [ ]:
def encode_corpus_now(desc="Encode corpus"):
    """Encode toàn bộ all_chunk_texts bằng trọng số model HIỆN TẠI."""
    emb = encode_texts(all_chunk_texts, batch_size=BATCH_SIZE, desc=desc)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return emb


def top_ctx(results, k=5):
    """Top-k ctx_id (dedup theo văn bản) từ results đã xếp hạng giảm dần."""
    out = []
    for r in results:
        c = r["ctx_id"]
        if c not in out:
            out.append(c)
        if len(out) >= k:
            break
    return out


def compute_metrics(results_dict, query_path, k=5):
    """Recall@5 + Precision đúng công thức scoring.py."""
    gt = json.load(open(query_path, encoding="utf-8"))
    recall5s, precs = [], []
    for qid, item in gt.items():
        if qid not in results_dict:
            continue
        gold = {str(g) for g in item["answer"]}
        pred = [str(c) for c in top_ctx(results_dict[qid]["results"], k=k)]
        inter = gold & set(pred)
        recall5s.append(len(inter) / len(gold))
        precs.append(len(inter) / len(pred) if pred else 0.0)
    n = len(recall5s)
    return (sum(recall5s) / n if n else 0.0, sum(precs) / n if n else 0.0)

## 6. Fallback: Brute-force retrieve (định nghĩa trước — dùng trong bước baseline nếu tắt HNSW)

In [ ]:
def retrieve_dataset_brute_force(query_path, corpus_emb, k_save=TOP_SAVE, desc="Retrieve"):
    """Fallback brute-force retrieval nếu HNSW disabled."""
    queries = json.load(open(query_path, encoding="utf-8"))
    qids = sorted(queries.keys())
    q_texts = [QUERY_INSTRUCT + queries[q]["question"] for q in qids]
    q_embs = encode_texts(q_texts, batch_size=QUERY_BATCH, desc=desc)
    q_embs_t = torch.from_numpy(q_embs).to(DEVICE)
    C = torch.from_numpy(np.asarray(corpus_emb)).to(DEVICE)

    out = {}
    k_save = min(k_save, corpus_emb.shape[0])
    for i in range(0, len(qids), Q_BATCH):
        block = (C @ q_embs_t[i:i + Q_BATCH].T).T
        vals, idx = torch.topk(block, k=k_save, dim=1, largest=True, sorted=True)
        vals = vals.float().cpu().numpy()
        idx = idx.cpu().numpy()
        for j, qid in enumerate(qids[i:i + Q_BATCH]):
            out[qid] = {
                "question": queries[qid]["question"],
                "results": [{"ctx_id": int(ctx_ids[ii]), "chunk": int(chunk_idx[ii]),
                             "score": round(float(vals[j, t]), 4)} for t, ii in enumerate(idx[j])],
            }
    del q_embs_t, C
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

## 6b. Baseline (trước khi fine-tune) — kiểm tra cache

In [ ]:
def _legal_state_meta_ok():
    meta_path = os.path.join(LEGAL_STATE, "corpus_emb.meta.json")
    if not os.path.exists(meta_path):
        print(f"[vietlegal-state] Không thấy {meta_path} — bỏ qua cache.")
        return None
    meta = json.load(open(meta_path))
    if meta.get("model") != MODEL_NAME:
        print(f"[vietlegal-state] Cache là của model khác — bỏ qua.")
        return None
    if meta.get("max_seq_len") != MAX_SEQ_LEN:
        print(f"[vietlegal-state] max_seq_len khác — bỏ qua.")
        return None
    if tuple(meta.get("signature", ())) != sig:
        print(f"[vietlegal-state] Corpus signature khác — bỏ qua.")
        return None
    return meta


def load_baseline_results(name):
    if _legal_state_meta_ok() is None:
        return None
    p = os.path.join(LEGAL_STATE, f"{name}.json")
    if os.path.exists(p):
        print(f"[vietlegal-state] Dùng kết quả truy vấn có sẵn: {p}")
        return json.load(open(p, encoding="utf-8"))
    return None


def load_baseline_corpus_emb():
    if _legal_state_meta_ok() is None:
        return None
    emb_path = os.path.join(LEGAL_STATE, "corpus_emb.npy")
    if os.path.exists(emb_path):
        print(f"[vietlegal-state] Load corpus embedding có sẵn: {emb_path}")
        return np.load(emb_path, mmap_mode="r")
    return None


t0 = time.time()

if SKIP_BASELINE:
    print(f"[baseline] SKIP_BASELINE=True — bỏ qua encode+retrieve baseline (tốn ~2-3h).")
    print(f"[baseline] Dùng BASE_RECALL_REF={BASE_RECALL_REF:.4f} có sẵn làm warmup_recall@5 mốc epoch 0.")
    baseline_warmup_r5, baseline_warmup_p = BASE_RECALL_REF, float("nan")
    baseline_train_r5, baseline_train_p = float("nan"), float("nan")
else:
    baseline_results = {}
    for name in ["warmup", "train"]:
        cached = load_baseline_results(name)
        if cached is not None:
            baseline_results[name] = cached

    base_corpus_emb = None
    if len(baseline_results) < 2:
        base_corpus_emb = load_baseline_corpus_emb()
        if base_corpus_emb is None:
            print("[vietlegal-state] Không có cache khớp — encode corpus bằng model GỐC (chưa fine-tune)...")
            base_corpus_emb = encode_corpus_now(desc="Encode corpus (baseline)")
        for name in ["warmup", "train"]:
            if name not in baseline_results:
                if USE_HNSW:
                    baseline_results[name] = retrieve_dataset_with_hnsw(QUERY_FILES[name], base_corpus_emb, desc=f"Retrieve {name} (baseline, HNSW)")
                else:
                    # Fallback brute-force (giữ hàm cũ nếu tắt HNSW)
                    baseline_results[name] = retrieve_dataset_brute_force(QUERY_FILES[name], base_corpus_emb, desc=f"Retrieve {name} (baseline)")

    del base_corpus_emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    baseline_warmup_r5, baseline_warmup_p = compute_metrics(baseline_results["warmup"], QUERY_FILES["warmup"])
    baseline_train_r5, baseline_train_p = compute_metrics(baseline_results["train"], QUERY_FILES["train"])

print(f"\nBaseline (chưa fine-tune) — {time.time()-t0:.0f}s")
print(f"  warmup      Recall@5={baseline_warmup_r5:.4f}  Precision={baseline_warmup_p:.4f}")
print(f"  train Recall@5={baseline_train_r5:.4f}  Precision={baseline_train_p:.4f}")
print(f"  Ngưỡng lưu checkpoint: Recall@5(warmup) >= {SAVE_THRESHOLD:.4f}")

## 7. Xây cặp huấn luyện

In [ ]:
train_data = json.load(open(QUERY_FILES["train"], encoding="utf-8"))
print(f"train: {len(train_data)} câu hỏi")

training_pairs = []
skipped = 0
for qid, item in train_data.items():
    question = item["question"]
    answers = item.get("answer", [])
    if not answers:
        skipped += 1
        continue
    for a in answers:
        try:
            cid = int(a)
        except (TypeError, ValueError):
            continue
        idxs = ctx_id_to_indices.get(cid)
        if not idxs:
            continue
        training_pairs.append((question, all_chunk_texts[idxs[0]]))

print(f"Training pairs: {len(training_pairs)}  (bỏ qua {skipped} câu không có đáp án hợp lệ)")
print("Ví dụ pair đầu tiên:")
print("  Query   :", training_pairs[0][0][:100])
print("  Positive:", training_pairs[0][1][:100])

## 8. Dataset + collate

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


def collate_fn(batch):
    queries = [QUERY_INSTRUCT + q for q, _ in batch]
    positives = [p for _, p in batch]
    return queries, positives


print(f"Dataset sẵn sàng: {len(training_pairs)} cặp, batch_size={TRAIN_BATCH_SIZE} "
      f"-> {len(training_pairs) // TRAIN_BATCH_SIZE} step/epoch")

## 9. Fine-tune (NUM_EPOCHS epoch cấu hình ở trên) — **dùng HNSW để retrieve**

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.2f}%)")
optimizer = AdamW(trainable_params, lr=LEARNING_RATE)
steps_per_epoch = max(1, len(training_pairs) // TRAIN_BATCH_SIZE)
scheduler = CosineAnnealingLR(optimizer, T_max=steps_per_epoch * NUM_EPOCHS)

training_log = [{
    "epoch": 0, "loss": None,
    "warmup_recall@5": baseline_warmup_r5, "warmup_precision": baseline_warmup_p,
    "train_recall@5": baseline_train_r5, "train_precision": baseline_train_p,
    "saved_checkpoint": False, "note": "baseline (chưa fine-tune)",
}]
best_saved_recall = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n{'='*70}\nEpoch {epoch}/{NUM_EPOCHS}\n{'='*70}")

    loader = DataLoader(PairDataset(training_pairs), batch_size=TRAIN_BATCH_SIZE,
                        shuffle=True, collate_fn=collate_fn, drop_last=True)

    model.train()
    total_loss = 0.0
    for step, (queries, positives) in enumerate(tqdm(loader, desc=f"Train epoch {epoch}")):
        q_emb = encode_forward(queries)
        p_emb = encode_forward(positives)

        sim = (q_emb @ p_emb.T) / TEMPERATURE
        labels = torch.arange(sim.size(0), device=sim.device)
        loss = F.cross_entropy(sim, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f"  step {step + 1}/{steps_per_epoch}: loss={total_loss / (step + 1):.4f}")

    avg_loss = total_loss / len(loader)

    # --- Eval sau epoch: encode lại toàn corpus + truy vấn **HNSW** + tính metrics ---
    print(f"[Epoch {epoch}] Encoding corpus...")
    corpus_emb = encode_corpus_now(desc=f"Encode corpus (epoch {epoch})")
    
    print(f"[Epoch {epoch}] Retrieving with {'HNSW' if USE_HNSW else 'brute-force'}...")
    if USE_HNSW:
        warmup_out = retrieve_dataset_with_hnsw(QUERY_FILES["warmup"], corpus_emb, desc=f"Retrieve warmup (epoch {epoch}, HNSW)")
        train_out = retrieve_dataset_with_hnsw(QUERY_FILES["train"], corpus_emb, desc=f"Retrieve train (epoch {epoch}, HNSW)")
    else:
        warmup_out = retrieve_dataset_brute_force(QUERY_FILES["warmup"], corpus_emb, desc=f"Retrieve warmup (epoch {epoch})")
        train_out = retrieve_dataset_brute_force(QUERY_FILES["train"], corpus_emb, desc=f"Retrieve train (epoch {epoch})")
    
    warmup_r5, warmup_p = compute_metrics(warmup_out, QUERY_FILES["warmup"])
    train_r5, train_p = compute_metrics(train_out, QUERY_FILES["train"])

    print(f"[Epoch {epoch}] loss={avg_loss:.4f} | warmup Recall@5={warmup_r5:.4f} Precision={warmup_p:.4f} "
          f"| train Recall@5={train_r5:.4f} Precision={train_p:.4f}")

    saved = False
    if warmup_r5 >= SAVE_THRESHOLD:
        saved = True
        if USE_LORA:
            ckpt_dir = os.path.join(MODEL_OUT_DIR, f"epoch{epoch}_recall{warmup_r5:.4f}")
            get_trainable_model().save_pretrained(ckpt_dir)   # chỉ lưu adapter LoRA (vài MB), không lưu full base
            print(f"  -> Vượt ngưỡng {SAVE_THRESHOLD:.4f} — đã lưu LoRA adapter: {ckpt_dir}")
        else:
            ckpt_path = os.path.join(MODEL_OUT_DIR, f"epoch{epoch}_recall{warmup_r5:.4f}.pth")
            torch.save(model.state_dict(), ckpt_path)
            print(f"  -> Vượt ngưỡng {SAVE_THRESHOLD:.4f} — đã lưu checkpoint: {ckpt_path}")
        if warmup_r5 > best_saved_recall:
            best_saved_recall = warmup_r5
            if USE_LORA:
                get_trainable_model().save_pretrained(os.path.join(MODEL_OUT_DIR, "best_adapter"))
            else:
                torch.save(model.state_dict(), os.path.join(MODEL_OUT_DIR, "best_model.pth"))
            json.dump(warmup_out, open(os.path.join(OUT_DIR, "warmup_finetuned_best.json"), "w", encoding="utf-8"),
                       ensure_ascii=False, separators=(",", ":"))
            json.dump(train_out, open(os.path.join(OUT_DIR, "train_finetuned_best.json"), "w", encoding="utf-8"),
                       ensure_ascii=False, separators=(",", ":"))
            print(f"  -> Điểm cao nhất tới hiện tại — cập nhật {'best_adapter' if USE_LORA else 'best_model.pth'}")

    training_log.append({
        "epoch": epoch, "loss": avg_loss,
        "warmup_recall@5": warmup_r5, "warmup_precision": warmup_p,
        "train_recall@5": train_r5, "train_precision": train_p,
        "saved_checkpoint": saved,
    })

    del corpus_emb, warmup_out, train_out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nHoàn tất huấn luyện.")

## 10. Tổng hợp kết quả theo epoch

In [ ]:
print("=" * 90)
print(f"{'Epoch':>6s} | {'Loss':>8s} | {'Warmup R@5':>11s} | {'Warmup Prec':>12s} | "
      f"{'Train R@5':>10s} | {'Saved':>6s}")
print("-" * 90)
for row in training_log:
    loss_str = f"{row['loss']:.4f}" if row["loss"] is not None else "  --  "
    print(f"{row['epoch']:>6d} | {loss_str:>8s} | {row['warmup_recall@5']:>11.4f} | "
          f"{row['warmup_precision']:>12.4f} | {row['train_recall@5']:>10.4f} | "
          f"{('YES' if row['saved_checkpoint'] else '-'):>6s}")
print("=" * 90)
print(f"Baseline (epoch 0) Recall@5={baseline_warmup_r5:.4f} | Ngưỡng lưu={SAVE_THRESHOLD:.4f} | "
      f"Best đạt được={best_saved_recall:.4f}" if best_saved_recall > 0 else
      f"Baseline (epoch 0) Recall@5={baseline_warmup_r5:.4f} | Ngưỡng lưu={SAVE_THRESHOLD:.4f} | "
      f"Không có epoch nào vượt ngưỡng — không checkpoint nào được lưu.")
print(f"\nRetrieval mode: {'HNSW' if USE_HNSW else 'Brute-force'}")

## 11. Lưu model cuối cùng + config + log

In [ ]:
if USE_LORA:
    final_adapter_dir = os.path.join(MODEL_OUT_DIR, "final_adapter")
    get_trainable_model().save_pretrained(final_adapter_dir)
    print(f"Đã lưu LoRA adapter cuối: {final_adapter_dir}")

    # Merge LoRA vào base rồi lưu full model — để các notebook retrieval khác
    # (load bằng AutoModel.from_pretrained thẳng, không biết PEFT) vẫn dùng được.
    merged_dir = os.path.join(MODEL_OUT_DIR, "final_model_merged")
    merged_model = get_trainable_model().merge_and_unload()
    merged_model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)
    print(f"Đã merge LoRA + lưu full model: {merged_dir}")
    final_model_path = merged_dir
else:
    final_model_path = os.path.join(MODEL_OUT_DIR, "final_model.pth")
    torch.save(model.state_dict(), final_model_path)
    tokenizer.save_pretrained(MODEL_OUT_DIR)
    print(f"Đã lưu model cuối: {final_model_path}")
    print(f"Đã lưu tokenizer : {MODEL_OUT_DIR}")

config_json = {
    "model_name": MODEL_NAME,
    "max_seq_len": MAX_SEQ_LEN,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "temperature": TEMPERATURE,
    "base_recall_ref": BASE_RECALL_REF,
    "min_improvement": MIN_IMPROVEMENT,
    "save_threshold": SAVE_THRESHOLD,
    "baseline_warmup_recall@5": baseline_warmup_r5,
    "best_warmup_recall@5": best_saved_recall,
    "has_saved_checkpoint": best_saved_recall > 0,
    "hnsw_enabled": USE_HNSW,
    "hnsw_m": HNSW_M,
    "hnsw_ef_construction": HNSW_EF_CONST,
    "hnsw_ef_search": HNSW_EF_SEARCH,
    "use_lora": USE_LORA,
    "lora_r": LORA_R if USE_LORA else None,
    "lora_alpha": LORA_ALPHA if USE_LORA else None,
    "lora_dropout": LORA_DROPOUT if USE_LORA else None,
    "lora_target_modules": LORA_TARGET_MODULES if USE_LORA else None,
}
json.dump(config_json, open(os.path.join(MODEL_OUT_DIR, "config.json"), "w"), indent=2, ensure_ascii=False)
json.dump({"training_log": training_log, "config": config_json},
          open(os.path.join(OUT_DIR, "finetuning_log.json"), "w"), indent=2, ensure_ascii=False)

print(f"\nĐã lưu config : {os.path.join(MODEL_OUT_DIR, 'config.json')}")
print(f"Đã lưu log    : {os.path.join(OUT_DIR, 'finetuning_log.json')}")
print(f"\n[HNSW Summary]")
print(f"  Enabled: {USE_HNSW}")
print(f"  M (max connections): {HNSW_M}")
print(f"  ef_construction: {HNSW_EF_CONST}")
print(f"  ef_search: {HNSW_EF_SEARCH}")
print(f"  Expected speedup on retrieval: 10-50x (depends on corpus size & parameters)")

## 12. ZIP `/kaggle/working` để tải về

In [ ]:
import zipfile

ZIP_PATH = os.path.join(OUT_DIR, "qwen3_finetuned_results_HNSW.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(OUT_DIR):
        for fn in fnames:
            if ".virtual" in fn or fn.endswith(".zip"):
                continue
            path = os.path.join(root, fn)
            zf.write(path, os.path.relpath(path, OUT_DIR))

print("Đã zip:", ZIP_PATH, "| size:", os.path.getsize(ZIP_PATH), "bytes")